# Ablation tour — portfolio-construction sweep

Visualisation layer over the saved outputs of the **portfolio-construction ablation
study** (`scripts/run_ablation.py` → `data/processed/ablation/`, `--grid full`). No
engine or selection logic runs here — regenerate the artifacts with:

    conda run -n ip2 python scripts/run_ablation.py --grid full

The study ([ABLATION_PLAN.md](../ABLATION_PLAN.md)) walks the full `2·2·3·2·3·3 = 216`
factorial over six construction knobs and scores each by **`sharpe_net_hedged`** — the
Sharpe of the market-beta-hedged, cost-net quarterly series over 2006–2025. Stages B/C
(OFAT + targeted interaction grids) were folded into the full factorial once Stage-A
timing showed engine runs were cheap, so every cell below is read straight off the
216-row grid.

In [ ]:
import json, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import polars as pl
from plotnine import (ggplot, aes, geom_col, geom_line, geom_point, geom_hline,
                      geom_vline, geom_tile, geom_boxplot, geom_text, geom_histogram,
                      facet_wrap, position_dodge, scale_fill_manual,
                      scale_fill_gradient2, scale_color_manual, labs, theme_bw,
                      theme, element_text)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config

cfg = config.load(PROJECT_ROOT / "config.yaml")
AB = PROJECT_ROOT / cfg["data"]["cache"] / "ablation"
pl.Config.set_tbl_rows(20); pl.Config.set_tbl_cols(-1)

results    = pl.read_ipc(AB / "results.feather")     # one row per config: 6 knobs + summary metrics
per_regime = pl.read_ipc(AB / "per_regime.feather")  # [config_id, regime, start, end, sharpe, ...]
meta       = json.loads((AB / "run_meta.json").read_text())

OBJ   = "sharpe_net_hedged"
KNOBS = ["strategic_allocation", "james_stein", "gross_leverage",
         "max_name_weight", "risk_aversion", "reselection_frequency_months"]
BASELINE = meta["baseline"]

# palette (matches backtest_tour) + a green for the winner
BLUE, ORANGE, LIGHT, GREEN = "#2c7fb8", "#d95f02", "#a6bddb", "#238b45"
# ordered categorical levels so numeric knob values sort correctly on a discrete axis
VALUE_LEVELS = ["equal", "ir_weighted", "true", "false", "0.02", "0.05",
                "1.0", "2.0", "3.0", "5.0", "24", "48", "240"]

def baseline_mask(df):
    m = pl.lit(True)
    for k, v in BASELINE.items():
        m = m & (pl.col(k) == v)
    return df.filter(m)

base_row   = baseline_mask(results)
win_row    = results.sort(OBJ, descending=True).head(1)
base_sharpe = float(base_row[OBJ][0])
win_sharpe  = float(win_row[OBJ][0])
print(f"grid '{meta['grid']}' | {meta['n_configs']} configs | built {meta['built_at']}")
print(f"baseline {OBJ} = {base_sharpe:+.3f}   winner {OBJ} = {win_sharpe:+.3f}")
print(f"selection cutoffs computed: {meta['cache_stats']['distinct_selection_cutoffs']} "
      f"| input builds: {meta['cache_stats']['distinct_input_builds']} "
      f"| wall {meta['wall_s']/60:.0f} min")

## 1. The objective landscape

One number per config: `sharpe_net_hedged`. Across the 216 cells it ranges roughly
`[−0.25, +0.55]` — construction choices alone swing net-hedged Sharpe by ~0.8, so the
knobs matter. The baseline (orange) sits mid-pack at **+0.31**; the grid-optimal config
(green) reaches **+0.55**. With ≈80 quarterly observations the Sharpe sampling error is
≈0.11 ([ABLATION_PLAN.md §D](../ABLATION_PLAN.md)), so read clusters and plateaus, not
third-decimal gaps.

In [ ]:
print(results.select(
    pl.col(OBJ).min().round(3).alias("min"),
    pl.col(OBJ).median().round(3).alias("median"),
    pl.col(OBJ).mean().round(3).alias("mean"),
    pl.col(OBJ).max().round(3).alias("max"),
    pl.col(OBJ).std().round(3).alias("std"),
))

hist_df = results.select(OBJ).to_pandas()
(ggplot(hist_df, aes(OBJ))
 + geom_histogram(bins=30, fill=BLUE, color="white")
 + geom_vline(xintercept=0, size=0.3)
 + geom_vline(xintercept=base_sharpe, linetype="dashed", color=ORANGE)
 + geom_vline(xintercept=win_sharpe, linetype="dashed", color=GREEN)
 + labs(title="Objective across all 216 configurations",
        subtitle=f"dashed: baseline ({base_sharpe:+.2f}, orange) vs winner ({win_sharpe:+.2f}, green)",
        x="sharpe_net_hedged", y="count")
 + theme_bw() + theme(figure_size=(9, 4)))

## 2. Leaderboard

Top configs ranked by the objective, with the §3 guardrails alongside. The whole top
of the table shares a signature: **`strategic_allocation = equal`, `james_stein = true`,
`reselection_frequency = 240`** (a single static selection — i.e. *no* dynamic
re-selection). The winner is the baseline with only `reselection_frequency` flipped
`24 → 240`. Guardrails stay clean at the top (net exposure well inside the `[−0.2, 0.2]`
band, shallow drawdowns); the `ra = 1.0` rows further down pay for their rank with
gross leverage > 1.5 and deeper drawdowns.

In [ ]:
guard = ["max_drawdown", "sharpe_net", "ir_vs_benchmark", "avg_tc_bps",
         "avg_gross_leverage", "avg_net_exposure", "avg_mkt_beta"]
show = ["strategic_allocation", "james_stein", "gross_leverage", "max_name_weight",
        "risk_aversion", "reselection_frequency_months", OBJ] + guard
lead = (results.sort(OBJ, descending=True)
        .with_columns(pl.col(pl.Float64).round(3))
        .select(show))
print("rank 1-15:")
lead.head(15)

## 3. Marginal effect of each knob

Each box is the objective's distribution over one knob value, marginalising across the
other five (all 216 configs). This is the robust "which lever moves the number" read:

- **`strategic_allocation`** is the single biggest lever — `equal` (mean ≈0.31)
  dominates `ir_weighted` (≈0.11).
- **`reselection_frequency`** is non-monotone: `240` (static) is best, `48` is *worst*
  — more frequent re-selection is not uniformly better.
- **`risk_aversion`** and **`max_name_weight`** trend up (higher `ra`, looser cap →
  higher mean); **`james_stein = true`** helps modestly.
- **`gross_leverage`** is nearly flat on average — at typical `risk_aversion` the risk
  penalty, not the gross cap, sets book size, so the cap is slack (see §5).

In [ ]:
long = (results.select([OBJ] + KNOBS)
        .with_columns([pl.col(k).cast(pl.Utf8) for k in KNOBS])
        .unpivot(index=OBJ, on=KNOBS, variable_name="knob", value_name="value")
        .to_pandas())
long["value"] = pd.Categorical(long["value"], categories=VALUE_LEVELS, ordered=True)
long["knob"]  = pd.Categorical(long["knob"], categories=KNOBS, ordered=True)
(ggplot(long, aes("value", OBJ))
 + geom_boxplot(fill=LIGHT, outlier_alpha=0.3)
 + geom_hline(yintercept=0, linetype="dashed", alpha=0.5)
 + facet_wrap("~knob", scales="free_x", ncol=3)
 + labs(title="Marginal effect of each knob (all 216 configs)",
        subtitle="each box = objective distribution over the other five knobs",
        x="", y="sharpe_net_hedged")
 + theme_bw()
 + theme(figure_size=(11, 6), subplots_adjust={"hspace": 0.35, "wspace": 0.2},
         strip_text=element_text(size=8), axis_text_x=element_text(size=8)))

## 4. One-factor-at-a-time response

The clean-attribution view: hold five knobs at baseline and vary the sixth (the dotted
line is the baseline objective). From the baseline, the single most valuable moves are
`reselection_frequency → 240` (+0.24), `risk_aversion → 5.0` (+0.08) and
`max_name_weight → 0.05` (+0.07); `ir_weighted` is catastrophic in isolation (−0.37).

Note the OFAT trap: from the baseline, `risk_aversion = 5.0` looks best — but the
*global* optimum keeps `risk_aversion = 2.0` and wins via `reselection_frequency`. OFAT
picks a local ridge; only the full factorial (§5) sees the interactions.

In [ ]:
ofat_frames = []
for k in KNOBS:
    m = pl.lit(True)
    for c in KNOBS:
        if c != k:
            m = m & (pl.col(c) == BASELINE[c])
    ofat_frames.append(
        results.filter(m)
        .select(pl.col(k).cast(pl.Utf8).alias("value"), pl.col(OBJ))
        .with_columns(knob=pl.lit(k))
    )
ofat = pl.concat(ofat_frames).to_pandas()
ofat["value"] = pd.Categorical(ofat["value"], categories=VALUE_LEVELS, ordered=True)
ofat["knob"]  = pd.Categorical(ofat["knob"], categories=KNOBS, ordered=True)
(ggplot(ofat, aes("value", OBJ, group="knob"))
 + geom_hline(yintercept=base_sharpe, linetype="dotted", color=ORANGE)
 + geom_line(color=BLUE)
 + geom_point(color=BLUE, size=2.5)
 + facet_wrap("~knob", scales="free_x", ncol=3)
 + labs(title="One-factor-at-a-time response from the baseline",
        subtitle=f"dotted = baseline objective ({base_sharpe:+.2f}); other five knobs held at baseline",
        x="", y="sharpe_net_hedged")
 + theme_bw()
 + theme(figure_size=(11, 6), subplots_adjust={"hspace": 0.35, "wspace": 0.2},
         strip_text=element_text(size=8), axis_text_x=element_text(size=8)))

## 5. Interactions

Mean objective over each 2-D slice (marginalising the other four knobs); the cell label
is the mean Sharpe. The key panel is **`risk_aversion × gross_leverage`**: at
`risk_aversion = 5` the row is flat — the risk penalty holds realised gross near 0.9, so
the gross cap never binds and leverage is a no-op. Only at `risk_aversion = 1` does
raising the cap change anything (and it pushes realised gross toward 3). This is exactly
why an OFAT sweep of leverage alone (§4, flat line) is misleading — leverage only bites
once `risk_aversion` is low enough to drive the book into the cap.

In [ ]:
def heat(xcol, ycol, title):
    g = (results.group_by(xcol, ycol)
         .agg(pl.col(OBJ).mean().alias("sharpe"))
         .with_columns(label=pl.col("sharpe").round(2).cast(pl.Utf8),
                       x=pl.col(xcol).cast(pl.Utf8), y=pl.col(ycol).cast(pl.Utf8))
         .to_pandas())
    g["x"] = pd.Categorical(g["x"], categories=VALUE_LEVELS, ordered=True)
    g["y"] = pd.Categorical(g["y"], categories=VALUE_LEVELS, ordered=True)
    return (ggplot(g, aes("x", "y", fill="sharpe"))
            + geom_tile(color="white")
            + geom_text(aes(label="label"), size=10)
            + scale_fill_gradient2(low="#b2182b", mid="#f7f7f7", high=GREEN,
                                   midpoint=float(results[OBJ].median()))
            + labs(title=title, x=xcol, y=ycol, fill="mean\nsharpe")
            + theme_bw() + theme(figure_size=(6, 4.5)))

heat("gross_leverage", "risk_aversion", "risk_aversion x gross_leverage")

In [ ]:
# the other two prioritised pairs from ABLATION_PLAN Stage C
print(heat("max_name_weight", "gross_leverage", "gross_leverage x max_name_weight"))
heat("james_stein", "strategic_allocation", "strategic_allocation x james_stein")

## 6. Guardrail scatter

Objective against the guardrail metrics, one point per config, coloured by
`risk_aversion`. The point is to catch configs that would win on Sharpe while breaking a
guardrail. Two things stand out: (i) the deep-drawdown / high-gross / high-turnover tail
is overwhelmingly **low `risk_aversion` (orange)** — the aggressive books; and (ii) the
best objective values are *not* the highest-leverage ones — Sharpe does not reward
piling on gross. Transaction cost is tiny throughout (a few bps), so cost is not the
differentiator at these turnover levels.

In [ ]:
gcols = ["avg_gross_leverage", "max_drawdown", "avg_tc_bps", "avg_turnover"]
gl = (results.select([OBJ, "risk_aversion"] + gcols)
      .with_columns(risk_aversion=pl.col("risk_aversion").cast(pl.Utf8))
      .unpivot(index=[OBJ, "risk_aversion"], on=gcols,
               variable_name="guardrail", value_name="value")
      .to_pandas())
gl["risk_aversion"] = pd.Categorical(gl["risk_aversion"],
                                     categories=["1.0", "2.0", "5.0"], ordered=True)
(ggplot(gl, aes("value", OBJ, color="risk_aversion"))
 + geom_point(alpha=0.6, size=1.6)
 + geom_hline(yintercept=0, linetype="dashed", alpha=0.4)
 + facet_wrap("~guardrail", scales="free_x", ncol=2)
 + scale_color_manual(values={"1.0": ORANGE, "2.0": "#7570b3", "5.0": GREEN})
 + labs(title="Objective vs guardrails (one point per config)",
        subtitle="low risk_aversion (orange) drives the leverage / drawdown / turnover tail",
        x="", y="sharpe_net_hedged", color="risk_aversion")
 + theme_bw()
 + theme(figure_size=(10, 6), subplots_adjust={"hspace": 0.3, "wspace": 0.2},
         legend_position="top"))

## 7. Per-regime robustness — is a winner a single-regime fluke?

`per_regime.feather` slices each config's net (cost-only) Sharpe by 2-year regime. The
global winner uses `reselection_frequency = 240`, i.e. one static regime over the whole
window, so it has no per-regime granularity to inspect. For a fair robustness read we
compare the **baseline** against the **best `reselection_frequency = 24` config**
(`equal, js=true, gl=3, mnw=0.05, ra=5`) — both resolve to the same ten 2-year regimes.
The tuned config is positive in more regimes and less negative in the bad ones, but both
still swing sign across regimes — a reminder that ~80 quarters carry real sampling error
and no config is uniformly good across every regime.

In [ ]:
rf24     = results.filter(pl.col("reselection_frequency_months") == 24)
best24   = rf24.sort(OBJ, descending=True).head(1)
base_id  = base_row["config_id"][0]
best24_id = best24["config_id"][0]
print(f"baseline      : {base_id}")
print(f"best (rf=24)  : {best24_id}  ->  {OBJ} = {float(best24[OBJ][0]):+.3f}")

sel = {base_id: "baseline", best24_id: "best (rf=24)"}
pr = (per_regime.filter(pl.col("config_id").is_in(list(sel)))
      .with_columns(cfg=pl.col("config_id").replace(sel))
      .to_pandas())
(ggplot(pr, aes("factor(regime)", "sharpe", fill="cfg"))
 + geom_col(position=position_dodge(width=0.8))
 + geom_hline(yintercept=0, size=0.3)
 + scale_fill_manual(values={"baseline": LIGHT, "best (rf=24)": BLUE})
 + labs(title="Per-regime Sharpe (net, cost-only): baseline vs best dynamic config",
        subtitle="rf=24 -> ten 2-year regimes (2006-2025); a robust winner stays positive across regimes",
        x="regime", y="per-regime Sharpe (net)", fill="")
 + theme_bw() + theme(figure_size=(10, 4), legend_position="top"))

## Takeaways

- **The dominant lever is `strategic_allocation = equal`** — `ir_weighted` roughly
  halves mean Sharpe and blows up drawdowns.
- **Static selection (`reselection_frequency = 240`) beats dynamic re-selection** on
  this window; the `48`-month cadence is the worst. Dynamic re-selection is buying
  adaptiveness the 2006–2025 sample does not reward — a finding worth carrying into
  [PORTFOLIO_PLAN.md](../PORTFOLIO_PLAN.md).
- **`gross_leverage` is near-irrelevant** unless `risk_aversion` is low enough for the
  cap to bind; the two must be read jointly (§5).
- **The grid winner** (`equal, js=true, gl=2, mnw=0.02, ra=2, rf=240`, Sharpe **+0.55**)
  is the baseline with only the re-selection cadence changed, and it clears every
  guardrail. Given the ≈0.11 Sharpe SE, treat the whole `equal / js=true / rf=240` band
  as the plateau rather than fixating on the single top cell.

These are exploratory reads over the saved grid; the canonical winner-selection and
`config.yaml` update happen in `scripts/run_backtest.py` per
[ABLATION_PLAN.md §D](../ABLATION_PLAN.md).